# Clinical AI Copilot - Quick Start Demo

This notebook demonstrates the basic usage of the Clinical AI Copilot system.

## Contents
1. EEG Signal Processing
2. Seizure Detection
3. Clinical Knowledge Retrieval
4. Multimodal Analysis

In [ ]:
# Imports
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.signal_processing.eeg_processor import EEGProcessingPipeline
from src.models.cnn_lstm import HybridCNNLSTM, SeizureDetectionConfig
from src.rag.vector_store import ClinicalVectorStore, Document

print("Imports successful!")

## 1. Generate Sample EEG Data

In [ ]:
# Generate realistic EEG data
def generate_sample_eeg(duration=1.0, sampling_rate=256, channels=16):
    num_samples = int(duration * sampling_rate)
    t = np.linspace(0, duration, num_samples)
    
    eeg_data = np.zeros((channels, num_samples))
    
    for ch in range(channels):
        # Alpha rhythm (10 Hz)
        alpha = 20 * np.sin(2 * np.pi * 10 * t)
        # Beta rhythm (20 Hz)
        beta = 10 * np.sin(2 * np.pi * 20 * t)
        # Noise
        noise = 5 * np.random.randn(num_samples)
        
        eeg_data[ch, :] = alpha + 0.5 * beta + noise
    
    return eeg_data.astype(np.float32)

eeg_chunk = generate_sample_eeg()
print(f"EEG data shape: {eeg_chunk.shape}")

# Visualize
plt.figure(figsize=(12, 6))
for i in range(4):  # Plot first 4 channels
    plt.plot(eeg_chunk[i, :] + i * 50, label=f'Channel {i+1}')
plt.xlabel('Samples')
plt.ylabel('Amplitude (µV)')
plt.title('Sample EEG Signals')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Process EEG with Signal Processing Pipeline

In [ ]:
# Initialize processor
processor = EEGProcessingPipeline(sampling_rate=256, channels=16)

# Process chunk
features, wavelets = processor.process_chunk(eeg_chunk)

print("Extracted Features:")
print(f"  PSD Features: {list(features['psd'].keys())}")
print(f"  Entropy Features: {list(features['entropy'].keys())}")
print(f"  Hjorth Features: {list(features['hjorth'].keys())}")

# Display band powers
print("\nBand Powers (average across channels):")
for band, power in features['psd'].items():
    if isinstance(power, np.ndarray):
        print(f"  {band}: {np.mean(power):.2f} µV²/Hz")

## 3. Seizure Detection

In [ ]:
# Create model
config = SeizureDetectionConfig()
model = HybridCNNLSTM(config)
model.eval()

# Prepare input (batch=1, windows=10, channels=16, samples=256)
eeg_windows = []
for _ in range(10):
    eeg_windows.append(generate_sample_eeg())

eeg_tensor = torch.tensor(np.array(eeg_windows), dtype=torch.float32).unsqueeze(0)

# Run inference
with torch.no_grad():
    output, attention = model(eeg_tensor, return_attention=True)
    probabilities = torch.softmax(output, dim=1)

# Display results
classes = ['Normal', 'Pre-ictal', 'Ictal']
probs = probabilities[0].numpy()

print("Seizure Detection Results:")
for class_name, prob in zip(classes, probs):
    print(f"  {class_name}: {prob*100:.2f}%")

# Visualize
plt.figure(figsize=(8, 4))
plt.bar(classes, probs * 100)
plt.ylabel('Probability (%)')
plt.title('Seizure Detection Probabilities')
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.show()

## 4. Clinical Knowledge Retrieval

In [ ]:
# Create vector store
vector_store = ClinicalVectorStore(use_memory=True)

# Add sample medical documents
docs = [
    Document(
        id="1",
        text="Epilepsy is characterized by recurrent seizures and abnormal EEG patterns.",
        metadata={'specialty': 'neurology', 'year': 2023}
    ),
    Document(
        id="2",
        text="Levetiracetam is an effective antiepileptic drug with minimal side effects.",
        metadata={'specialty': 'neurology', 'year': 2022}
    ),
    Document(
        id="3",
        text="EEG monitoring is essential for seizure detection and classification.",
        metadata={'specialty': 'neurology', 'year': 2023}
    )
]

vector_store.add_documents(docs)

# Search for relevant information
query = "seizure treatment options"
results = vector_store.search(query, top_k=2)

print(f"Search results for: '{query}'\n")
for doc, score in results:
    print(f"Score: {score:.4f}")
    print(f"Text: {doc.text}")
    print(f"Metadata: {doc.metadata}")
    print()

## 5. Complete Analysis Pipeline

Putting it all together for a complete patient analysis.

In [ ]:
def analyze_patient_eeg(eeg_data, patient_symptoms):
    """
    Complete analysis pipeline
    
    Args:
        eeg_data: EEG data array
        patient_symptoms: Patient symptoms text
        
    Returns:
        Analysis results
    """
    # 1. Process EEG
    features, _ = processor.process_chunk(eeg_data)
    anomalies = processor.detect_anomalies(features)
    
    # 2. Detect seizures
    # (Simplified - in practice, would use full 10-second windows)
    seizure_prob = 0.15  # Dummy value
    
    # 3. Retrieve clinical knowledge
    clinical_context = vector_store.search(patient_symptoms, top_k=3)
    
    # 4. Generate report
    report = {
        'seizure_probability': seizure_prob,
        'anomalies_detected': sum(anomalies.values()),
        'band_powers': {
            'alpha': np.mean(features['psd']['alpha']),
            'beta': np.mean(features['psd']['beta']),
            'theta': np.mean(features['psd']['theta'])
        },
        'clinical_recommendations': [
            doc.text for doc, score in clinical_context
        ]
    }
    
    return report

# Analyze
patient_symptoms = "recurrent seizures and loss of consciousness"
report = analyze_patient_eeg(eeg_chunk, patient_symptoms)

print("=" * 60)
print("CLINICAL ANALYSIS REPORT")
print("=" * 60)
print(f"\nSeizure Probability: {report['seizure_probability']*100:.1f}%")
print(f"Anomalies Detected: {report['anomalies_detected']}")
print(f"\nBand Powers:")
for band, power in report['band_powers'].items():
    print(f"  {band.capitalize()}: {power:.2f} µV²/Hz")
print(f"\nClinical Recommendations:")
for i, rec in enumerate(report['clinical_recommendations'], 1):
    print(f"  {i}. {rec}")
print("\n" + "=" * 60)

## Summary

This notebook demonstrated:
1. ✅ EEG signal generation and visualization
2. ✅ Signal processing (filtering, feature extraction)
3. ✅ Seizure detection with deep learning
4. ✅ Clinical knowledge retrieval with RAG
5. ✅ Complete analysis pipeline

Next steps:
- Explore real EEG data from medical databases
- Fine-tune models on your specific dataset
- Integrate with hospital EEG systems
- Deploy as API for production use